# Hello World — Claude Managed Agents (CLI)

The same minimal end-to-end flow as [`01-basics/`](../01-basics/), but driven entirely from the terminal with Anthropic's **`ant`** CLI instead of the Python SDK:

```
Environment (once) → Agent (once) → Session (every run) → Stream events
```

| Object | Lifecycle | Purpose |
|--------|-----------|----------|
| **Environment** | Create once | Sandboxed container where tools run (bash, files, code) |
| **Agent** | Create once | Versioned config: model, system prompt, tools |
| **Session** | Create per run | Links agent + environment; Anthropic runs the loop |

### CLI for the control plane, SDK for the data plane

The split this notebook teaches: **agents and environments are static resources** you define as version-controlled YAML and apply with `ant` (control plane). **Sessions are dynamic** — created per run and streamed (data plane). Both hit the same API; the difference is where the call lives.

> **In production:** persist `environment.id` and `agent.id` — don't recreate them on every run.

## 1. Setup

The CLI exposes every Claude API resource as a shell subcommand. Beta resources (agents, environments, sessions) live under the `beta:` prefix, and the CLI sets the right `anthropic-beta` header automatically.

First, make sure `ant` is installed.

In [ ]:
%%bash
# If `ant` isn't installed yet, install it (pick one):
#   macOS:        brew install anthropics/tap/ant
#   Linux / WSL:  download a release from github.com/anthropics/anthropic-cli/releases
#   From source:  go install github.com/anthropics/anthropic-cli/cmd/ant@latest
ant --version || echo "ant not found — install it using one of the methods above"


### Authentication

`ant` is a separate process from this notebook, so unlike the Python SDK it does **not** read the root `.env`. The cell below loads that `.env` into `os.environ` (every `!ant` call inherits it) and clears Jupyter's `FORCE_COLOR`, which would otherwise make `ant` wrap its JSON/YAML in ANSI color codes and break parsing.

It resolves credentials the same way the SDKs do (first match wins): `ANTHROPIC_API_KEY`, then `ANTHROPIC_AUTH_TOKEN`, then an `ant auth login` profile.

> **Trap:** profiles are only consulted when no API key is set. A stale exported `ANTHROPIC_API_KEY` silently overrides every profile. `ant auth status` shows which source won.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# `ant` is a separate process from this notebook, so two bits of setup:
#  - it doesn't read the root .env — load the API key into os.environ, which
#    every `!ant` call below inherits;
#  - Jupyter sets FORCE_COLOR=1, which makes ant wrap its JSON/YAML output in
#    ANSI color codes and break parsing — clear it so the output stays plain.
load_dotenv(find_dotenv())
os.environ.pop("FORCE_COLOR", None)
os.environ.pop("CLICOLOR_FORCE", None)
os.environ["NO_COLOR"] = "1"
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not found — add it to the root .env or run `ant auth login`."
)
print("Credential loaded, color disabled — ant is ready")

## 2. Create an Environment

The environment is the **sandboxed container** where tools execute. We define it as version-controlled YAML — the recommended control-plane flow — then apply it with `ant`.

- `type: cloud` — Anthropic manages the infra
- `networking.type: unrestricted` — allows outbound internet access

**Create this once and reuse the ID.**

In [ ]:
%%writefile env.yaml
name: basics-env-cli
config:
  type: cloud
  networking:
    type: unrestricted


Apply it. `--transform id -r` extracts just the `id` field as a bare string (no quotes), which we capture into a Python variable so later cells can reuse it — the CLI analog of storing `environment.id`.

In [ ]:
# Pipe the YAML in via stdin; capture the new environment's id.
_env = !ant beta:environments create --transform id -r < env.yaml  # type: ignore
env_id = _env[0]
print("Environment ID:", env_id)

## 3. Create an Agent

The agent is a **persisted, versioned config**: model, system prompt, and tools. `agent_toolset_20260401` is Anthropic's prebuilt toolset (bash, file ops, code execution, web search/fetch) — all running inside the environment container.

Every update creates a new **immutable version**, so sessions can pin to a specific version and never break.

In [ ]:
%%writefile agent.yaml
name: Hello World Agent (CLI)
model: claude-opus-4-7
system: |
  You are a helpful assistant. Keep your answers concise.
tools:
  - type: agent_toolset_20260401
    default_config:
      enabled: true


We capture the full JSON response here so we can grab **both** the `id` and the `version` — sessions pin to an explicit version.

In [ ]:
import json

# --format json prints the full object; capture id + version for explicit pinning.
_agent = !ant beta:agents create --format json < agent.yaml  # type: ignore
agent = json.loads("".join(_agent))
agent_id, agent_version = agent["id"], agent["version"]
print("Agent ID :", agent_id)
print("Version  :", agent_version)

## 4. Create a Session

A session is a **single run** — it links the agent to the environment. This is the data plane: created per run, streamed, and driven by events.

We pin to the exact `agent.version` so future agent updates don't affect this session, and capture the session id into `session_id` for the streaming cell below.

In [ ]:
import json

# Pin the agent to its exact version: {"type": "agent", "id": ..., "version": ...}
agent_ref = json.dumps({"type": "agent", "id": agent_id, "version": agent_version})

_session = !ant beta:sessions create --agent '{agent_ref}' --environment-id {env_id} --title "Hello World CLI session" --transform id -r  # type: ignore
session_id = _session[0]
print("Session ID:", session_id)

## 5. Stream Events

This is the **stream-first pattern** — the headline rule of Managed Agents:

1. **Open the stream first** (`ant beta:sessions:events stream`) — before sending anything
2. **Then send the user message** (`ant beta:sessions:events send`) — triggers the agent loop
3. **Read events as they arrive** — print `agent.message` text until the session goes idle

If you send before opening the stream, you'll miss early events.

| Event type | Meaning |
|---|---|
| `agent.message` | Claude's response text |
| `session.status_idle` | Agent finished its turn |
| `session.status_terminated` | Session is done |

Unlike the create steps above (one-shot `!ant` calls), streaming needs to read events **live**, so we drive `ant` from Python with `subprocess`. We read `--format yaml` because `--format jsonl` buffers until the stream ends.

In [ ]:
import subprocess, json

def stream_reply(session_id: str, message: str) -> None:
    """Stream-first: open the event stream BEFORE sending, then read events live.

    This is the one step that needs real parsing rather than a one-liner:
    `--format jsonl` buffers until the stream ends, so we read `--format yaml`,
    where each event's `content:` block precedes its `type:` line (so we buffer
    the text and print it only once we learn it was an agent.message), and
    multi-line replies arrive as a `- text: |-` block scalar.
    """
    # 1. Open the stream FIRST (stream-before-send) — miss this and early events vanish.
    stream = subprocess.Popen(
        ["ant", "beta:sessions:events", "stream", "--session-id", session_id, "--format", "yaml"],
        stdout=subprocess.PIPE, text=True, bufsize=1,
    )
    # 2. Send the user message — this triggers the agent loop.
    subprocess.run(
        ["ant", "beta:sessions:events", "send", "--session-id", session_id],
        input=json.dumps({"events": [{"type": "user.message",
                "content": [{"type": "text", "text": message}]}]}),
        stdout=subprocess.DEVNULL, text=True, check=True,
    )
    # 3. Read events until the session goes idle.
    print("Agent: ", end="", flush=True)
    pending, block, buf, indent0 = None, False, [], None
    for raw in stream.stdout:
        line = raw.rstrip("\n")
        if block:  # collecting a block scalar; blank lines belong to it
            if line.strip() == "":
                buf.append(""); continue
            ind = len(line) - len(line.lstrip())
            if indent0 is None or ind >= indent0:
                indent0 = ind if indent0 is None else indent0
                buf.append(line[indent0:]); continue
            pending, block = "\n".join(buf).rstrip("\n"), False  # dedent → block ends
        if line.strip().startswith("- text:"):
            val = line.split("- text:", 1)[1].strip()
            if val in ("|", "|-", ">", ">-", ""):
                block, buf, indent0 = True, [], None
            else:
                pending = val
        elif line == "type: agent.message":
            if pending is not None:
                print(pending, end="", flush=True)
            pending = None
        elif line == "type: user.message":
            pending = None
        elif line in ("type: session.status_idle", "type: session.status_terminated"):
            break
    stream.terminate()
    print()

stream_reply(session_id, "Say hello and tell me one fun fact about octopuses.")

## Summary

You just ran the full Claude Managed Agents flow from the terminal:

```
ant beta:environments create < env.yaml   →  store the environment id
ant beta:agents create < agent.yaml        →  store agent id + version
ant beta:sessions create --agent ...        →  new session per run
ant beta:sessions:events stream             →  open stream FIRST
ant beta:sessions:events send               →  send message, triggers loop
read loop                                   →  handle agent.message until idle
```

This mirrors [`01-basics/`](../01-basics/) line for line — the Python SDK version does the same thing in code. Use whichever fits: the CLI for ad-hoc control-plane work and version-controlled YAML in CI, the SDK for application code that drives sessions.

### CLI cheatsheet

```bash
ant beta:agents list --transform '{id,name,model}' --format jsonl   # inspect agents
ant beta:sessions:events list --session-id "$SID" --transform 'content.0.text' -r
ant beta:agents update --agent-id "$AGENT_ID" --version N < agent.yaml   # new version
ant --help                                                          # explore
```

### Next steps
- **02-multi-turn** — keep the session alive and send follow-up messages
- **03-tools** — use bash and file tools inside the container
- **04-mcp** — connect external MCP servers to the agent